# Train our own model

Thin driver: mounts Drive, downloads the dataset, clones the repo, builds the
manifest (reusing `data.coco_split`), then calls `train_model.py` / `export_model.py`
as subprocesses -- no training logic lives in this notebook.

Checkpoints and the exported model are written to Drive (`RUN_ROOT`), same pattern
as the transformer-pytorch repo: `latest.pt` (full resumable state) and `best.pt`
(lightest, only overwritten when validation mAP@0.5 improves).

## 1. Mount Drive

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

RUN_ROOT = "/content/drive/MyDrive/yolo-edge-quantization"
# one Drive folder per run: reusing an existing name RESUMES that run from its latest.pt
RUN_NAME = "run04_long"

## 2. Download the dataset

In [ ]:
!pip install -q roboflow

from google.colab import userdata
from roboflow import Roboflow

rf = Roboflow(api_key=userdata.get("ROBOFLOW_API_KEY"))
project = rf.workspace("efe-tatar").project("yolo-to-coco-traffic-signs-pmo2q")
dataset = project.version(1).download("coco")

datasetDirectory = dataset.location
print(datasetDirectory)

## 3. Clone the repo and install its requirements

In [ ]:
import os

repoDirectory = "/content/yolo-edge-quantization"
if not os.path.isdir(repoDirectory):
    !git clone -q https://github.com/efe-tatar/yolo-edge-quantization.git {repoDirectory}
else:
    !git -C {repoDirectory} pull -q

!pip install -q -r {repoDirectory}/requirements.txt

import sys
sys.path.insert(0, repoDirectory)

## 4. Build the manifest

Same `data.coco_split` functions as `dataset_split.ipynb` -- nothing reimplemented
here. Also saves `manifest.json` + `classes.json` to Drive, so this exact split is
on record alongside the checkpoints it produced.

In [ ]:
import json

from data.coco_split import pool_boxes, redistribute, build_manifest

splitNames = ["train", "valid", "test"]
cocoBySplit = {}
for splitName in splitNames:
    with open(os.path.join(datasetDirectory, splitName, "_annotations.coco.json")) as jsonFile:
        cocoBySplit[splitName] = json.load(jsonFile)

boxes = pool_boxes(cocoBySplit)

classNames = sorted({box["class"] for box in boxes})
classToIndex = {className: i for i, className in enumerate(classNames)}
numClasses = len(classNames)
print(numClasses, "classes")

setOfPhoto, trainOnlyClasses = redistribute(boxes, min_photos=4, seed=42)
manifest = build_manifest(boxes, setOfPhoto, classToIndex, datasetDirectory)

for setName in ["train", "valid", "test"]:
    print(setName, ":", len(manifest[setName]), "images")

runDir = f"{RUN_ROOT}/{RUN_NAME}"
os.makedirs(runDir, exist_ok=True)

manifestPath = f"{runDir}/manifest.json"
with open(manifestPath, "w") as f:
    json.dump(manifest, f)

with open(f"{runDir}/classes.json", "w") as f:
    json.dump(classNames, f)

print("saved manifest to", manifestPath)

## 5. Train

`--thresholds 32 80` -- derived from this dataset's real box-size percentiles (p50=29px, p90=81px; see dataset_split.ipynb section 8), not a generic heuristic. Gives stride32 ~10% of boxes instead of ~2%, stride16 ~35% instead of ~15%.

Defaults since run02: AdamW (weight decay 0.01), 3-epoch warmup + cosine decay, training augmentation (scale/translate + colour jitter, no flips), 2 DataLoader workers -- see `train_model.py --help`. Pass `--no-augment` to reproduce run01's setup.

Since run03: center sampling (`--center-radius 1`: the cells around an object's center cell, inside its box, are positives too -- `--center-radius 0` reproduces run02) and val mAP every 5 epochs (`--map-every 5`, always on the last epoch).

Every epoch line ends with its timing (train / val loss / val mAP) and an ETA, and is also appended to `runDir/train_log.txt` on Drive, so the log survives a dropped session.
Reruns resume automatically from `runDir/latest.pt`.

**`DISCONNECT_WHEN_DONE`**: when training ends (finished *or* crashed), flush Drive and release the runtime, so an idle GPU doesn't keep burning compute units overnight. Sections 6-8 then need a new runtime -- a CPU one is enough for them.

In [ ]:
epochs = 100
batchSize = 16
DISCONNECT_WHEN_DONE = True

!python {repoDirectory}/train_model.py --manifest {manifestPath} --out-dir {runDir} --num-classes {numClasses} --strides 8 16 32 --thresholds 32 80 --epochs {epochs} --batch-size {batchSize}

if DISCONNECT_WHEN_DONE:
    from google.colab import runtime
    drive.flush_and_unmount()  # make sure the last checkpoint + log are written to Drive first
    runtime.unassign()

## 6. Check the result

In [ ]:
import torch

best = torch.load(f"{runDir}/best.pt", map_location="cpu", weights_only=False)
print("best epoch:", best["epoch"])
print("val mAP@0.5:", best["val_map"])

## 7. Export to ONNX

In [ ]:
onnxPath = f"{runDir}/model.onnx"

!python {repoDirectory}/export_model.py --checkpoint {runDir}/best.pt --num-classes {numClasses} --out {onnxPath}

## 8. Visualize predictions on a handful of test images

A sanity check, not a metric. Predictions in red, ground truth in green, on the same image.

One figure per confidence threshold in `confThresholds`, always the same 6 test images (fixed seed), so runs compare image for image. Each figure is saved to `runDir` on Drive as `conf03.png` etc. and downloaded to your computer -- copy them into `results/<run name>/` in the repo.

After a `DISCONNECT_WHEN_DONE` training run, rerun sections 1-4 first (a CPU runtime is enough).

In [ ]:
import random

import matplotlib.pyplot as plt
import torchvision.transforms as transforms
from google.colab import files

from data.dataset import DetectionDataset
from models.yolo import YOLO
from train.decode import decode_predictions
from train.trainer import load_checkpoint
from train.visualize import draw_detections

imgSize = 320  # must match what train_model.py was run with
strides = (8, 16, 32)
confThresholds = (0.3, 0.5, 0.7)

with open(f"{runDir}/classes.json") as f:
    classNamesLoaded = json.load(f)

model = YOLO(numClasses)
state = load_checkpoint(f"{runDir}/best.pt", model, device="cpu")
model.eval()
print("best.pt epoch:", state["epoch"], "| val mAP@0.5:", state.get("val_map"))

testDataset = DetectionDataset(manifest["test"], img_size=imgSize)

random.seed(0)
sampleIndices = random.sample(range(len(testDataset)), min(6, len(testDataset)))

# one forward pass per image, reused for every threshold
samples = []
for idx in sampleIndices:
    imageTensor, gtBoxes, gtLabels = testDataset[idx]
    with torch.no_grad():
        preds = model(imageTensor.unsqueeze(0))
    samples.append((imageTensor, gtBoxes, gtLabels, preds))

for confThresh in confThresholds:
    figure, axes = plt.subplots(2, 3, figsize=(15, 10))
    for axis, (imageTensor, gtBoxes, gtLabels, preds) in zip(axes.flat, samples):
        decoded = decode_predictions(preds, strides, conf_thresh=confThresh, iou_thresh=0.5)
        predBoxes, predScores, predLabels = decoded[0]

        image = transforms.ToPILImage()(imageTensor)
        image = draw_detections(image, gtBoxes, gtLabels, class_names=classNamesLoaded, color="green")
        image = draw_detections(image, predBoxes, predLabels, scores=predScores,
                                 class_names=classNamesLoaded, color="red")

        axis.imshow(image)
        axis.axis("off")
        axis.set_title(f"{len(predBoxes)} predicted, {len(gtBoxes)} ground truth")

    figure.suptitle(f"{RUN_NAME} -- best.pt (epoch {state['epoch']}), conf {confThresh}")
    plt.tight_layout()

    imagePath = f"{runDir}/conf{round(confThresh * 10):02d}.png"
    figure.savefig(imagePath, dpi=100, bbox_inches="tight")
    plt.show()
    files.download(imagePath)
    print("saved + downloading:", imagePath)

## 9. Random test pictures

Section 8 always shows the same 6 images, so runs compare image for image. This cell
draws a **new random set every time it runs**, to see how the model behaves beyond
those six. Needs section 8 to have run first (it reuses `model`, `testDataset`, ...).

- `seed = None` -> new images each run; the chosen indices are printed, so an
  interesting set can be reproduced by putting them in `indices`.
- `classAgnosticNms = True` merges overlapping boxes of different classes (what a
  deployed detector would do); `False` matches section 8 and the mAP evaluation.
- `saveAndDownload = True` saves the figure to `runDir` and downloads it.

In [ ]:
nImages = 9
confThresh = 0.5
classAgnosticNms = False
seed = None              # an int to reproduce a draw
indices = None           # or a list of test indices printed by an earlier run
saveAndDownload = False

if indices is None:
    rng = random.Random(seed)
    indices = rng.sample(range(len(testDataset)), min(nImages, len(testDataset)))
print("test indices:", indices)

nCols = 3
nRows = (len(indices) + nCols - 1) // nCols
figure, axes = plt.subplots(nRows, nCols, figsize=(5 * nCols, 5 * nRows), squeeze=False)

for axis in axes.flat:
    axis.axis("off")

for axis, idx in zip(axes.flat, indices):
    imageTensor, gtBoxes, gtLabels = testDataset[idx]
    with torch.no_grad():
        preds = model(imageTensor.unsqueeze(0))
    predBoxes, predScores, predLabels = decode_predictions(
        preds, strides, conf_thresh=confThresh, iou_thresh=0.5, class_agnostic_nms=classAgnosticNms
    )[0]

    image = transforms.ToPILImage()(imageTensor)
    image = draw_detections(image, gtBoxes, gtLabels, class_names=classNamesLoaded, color="green")
    image = draw_detections(image, predBoxes, predLabels, scores=predScores,
                             class_names=classNamesLoaded, color="red")

    axis.imshow(image)
    axis.set_title(f"test #{idx}: {len(predBoxes)} predicted, {len(gtBoxes)} ground truth")

nms = "class-agnostic NMS" if classAgnosticNms else "per-class NMS"
figure.suptitle(f"{RUN_NAME} -- best.pt (epoch {state['epoch']}), conf {confThresh}, {nms}")
plt.tight_layout()

if saveAndDownload:
    imagePath = f"{runDir}/random_{'_'.join(map(str, indices))}.png"
    figure.savefig(imagePath, dpi=100, bbox_inches="tight")
    files.download(imagePath)
    print("saved + downloading:", imagePath)
plt.show()